In [11]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
from torchvision.datasets import ImageFolder

dataset_path = "/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2"

full_dataset = ImageFolder(dataset_path, transform=transform)

class_names = full_dataset.classes
print("All classes:", class_names)

valid_classes = ['Au', 'Tp']
valid_indices = [class_names.index(cls) for cls in valid_classes]

filtered_samples = [
    (path, 0 if label == valid_indices[0] else 1)
    for path, label in full_dataset.samples
    if label in valid_indices
]

from torch.utils.data import Dataset

class BinaryCASIA(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = full_dataset.loader(path)
        image = self.transform(image)
        return image, label

dataset = BinaryCASIA(filtered_samples, transform)

print("Total images after filtering:", len(dataset))

All classes: ['Au', 'CASIA 2 Groundtruth', 'Tp']
Total images after filtering: 12614


In [4]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 8829
Validation: 1892
Test: 1893


In [5]:
model = timm.create_model('vit_base_patch16_224', pretrained=True)
model.head = nn.Linear(model.head.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [6]:
num_epochs = 30
best_val_acc = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    model.eval()
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.numpy())
    
    val_acc = accuracy_score(val_labels, val_preds)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} "
          f"Val Accuracy: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vit_model.pth")

100%|██████████| 276/276 [07:42<00:00,  1.68s/it]


Epoch [1/30] Train Loss: 0.5923 Val Accuracy: 0.7357


100%|██████████| 276/276 [06:07<00:00,  1.33s/it]


Epoch [2/30] Train Loss: 0.4762 Val Accuracy: 0.6697


100%|██████████| 276/276 [06:06<00:00,  1.33s/it]


...

Epoch [28/30] Train Loss: 0.1953 Val Accuracy: 0.6897


100%|██████████| 276/276 [06:03<00:00,  1.32s/it]


Epoch [29/30] Train Loss: 0.1830 Val Accuracy: 0.6882


100%|██████████| 276/276 [06:05<00:00,  1.33s/it]


Epoch [30/30] Train Loss: 0.1881 Val Accuracy: 0.6644


In [7]:
model.load_state_dict(torch.load("best_vit_model.pth"))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:,1]
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)
mce = 1 - accuracy

print("\nFinal Test Metrics:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("AUC:", auc)
print("MCE:", mce)


Final Test Metrics:
Accuracy: 0.7680929741151611
Precision: 0.6766666666666666
Recall: 0.8044914134742405
F1 Score: 0.7350633675316838
AUC: 0.8160001953597411
MCE: 0.2319070258848389


In [8]:
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[845 291]
 [148 609]]
